# Índices de mercado: quem venceu a inflação, em 2026 e em 15 anos?

Análise de retorno e risco de 10 índices que representam as principais classes de ativos de um investidor brasileiro, mais o S&P 500 convertido para reais. A lógica está em `src/indices/`; aqui cada etapa é executada e explicada.

**Perguntas:** 1) quanto cada classe rendeu no ano e em 12 meses, em termos nominais e reais? 2) qual foi o risco para chegar lá (volatilidade, maior queda)? 3) o retorno compensou o risco frente ao CDI? 4) e em 15 anos, como se sai uma carteira diversificada contra o CDI, a inflação e a meta IPCA + 4% a.a.?

In [1]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.precision", 2)
from indices import carteira, coleta, historico, metricas

## 1. Dados
A base mensal vem de relatório de mercado; CDI, IPCA e dólar são baixados da API pública do Banco Central (SGS) e substituem a base quando a API responde. O CDI chega como taxa **diária** e é encadeado para o mês; o dólar vira variação do último valor de cada mês.

**S&P 500:** a base traz o índice **em dólar** (price return), sem o efeito do câmbio. Para quem investe a partir do Brasil, a versão em reais é derivada: `(1 + S&P 500) × (1 + dólar) − 1`. As duas aparecem lado a lado, e o Sharpe do S&P em dólar não é calculado, porque compará-lo com o CDI misturaria moedas.

In [2]:
longo = coleta.montar()
largo = coleta.em_colunas(longo)
print("fontes:", sorted(longo.fonte.unique()), "| preliminares:", longo[longo.previa][["indice", "mes"]].values.tolist())
largo

API do BCB indisponível (ProxyError); usando só a base.
fontes: ['base'] | preliminares: [['IPCA', Period('2026-10', 'M')]]


indice,IBOV,S&P 500 (US$),S&P 500 (R$),IMA-B,CDI,IFIX,IRF-M,IPCA,IHFA,Dólar,BTC
mes,,,,,,,,,,,
2025-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.09,NaN,NaN,NaN
2025-11,6.37,0.13,-0.81,2.04,1.05,1.86,1.67,0.18,1.38,-0.94,-17.43
2025-12,1.29,-0.05,3.11,0.31,1.22,3.14,0.30,0.33,0.54,3.16,-1.44
2026-01,12.56,1.37,-3.65,1.00,1.16,2.27,1.96,0.33,2.23,-4.95,-15.66
2026-02,4.09,-0.87,-2.40,1.79,1.00,1.32,0.99,0.70,1.33,-1.54,-15.18
2026-03,-0.70,-5.09,-3.80,0.17,1.21,-1.06,-0.59,0.88,-3.42,1.36,3.02
2026-04,-0.08,10.42,5.54,1.81,1.09,1.53,1.24,0.67,2.05,-4.42,7.67
2026-05,-7.22,5.15,6.59,0.31,1.07,-1.33,0.68,0.58,0.62,1.37,-2.56
2026-06,-1.01,-1.06,1.28,-1.04,1.12,-1.21,0.69,0.16,0.51,2.37,-18.32


## 2. Retorno e risco
**Retornos se compõem, não se somam:** 1% + 1% em dois meses dá 2,01%, não 2%. Retorno real = (1 + nominal) / (1 + IPCA) − 1. Volatilidade anualizada = desvio-padrão mensal × √12. Sharpe = excesso médio sobre o CDI ÷ desvio do excesso × √12.

In [3]:
res = metricas.resumo(largo, 2026)
res

,indice,no_ano_2026,real_no_ano,ult_12m,real_12m,vol_anual,sharpe_12m,max_drawdown_12m,meses_positivos
0,CDI,11.68,7.09,14.23,8.98,0.25,NaN,0.00,12
1,S&P 500 (US$),10.74,6.19,10.83,5.74,13.31,NaN,-5.92,5
2,S&P 500 (R$),9.11,4.63,11.59,6.47,12.72,-0.13,-9.53,6
3,IMA-B,8.98,4.50,11.54,6.42,4.33,-0.53,-1.04,10
4,IRF-M,7.63,3.21,9.75,4.71,3.56,-1.10,-1.52,10
5,IHFA,5.81,1.47,7.85,2.90,5.65,-0.98,-3.42,9
6,IPCA,4.28,NaN,4.82,NaN,1.21,NaN,-0.32,11
7,IBOV,4.22,-0.06,12.29,7.13,20.63,0.01,-11.05,6
8,Dólar,-1.47,-5.52,0.69,-3.94,10.23,-1.19,-9.33,6
9,IFIX,-3.41,-7.38,1.47,-3.19,6.38,-1.85,-7.21,5


![no ano](../outputs/no_ano.png)

## 3. Evolução no ano

![evolução](../outputs/evolucao_ano.png)

## 4. Mês a mês

![mapa](../outputs/mapa_mensal.png)

## 5. Meta IPCA + 4% ao ano
Meta comum em planejamento de longo prazo. A taxa anual vira mensal por (1 + 4%)^(1/12) − 1 e é encadeada com o IPCA de cada mês.

In [4]:
ano = largo[largo.index.year == 2026]
meta = metricas.acumulado(metricas.benchmark_ipca_mais(ano, 4.0))
print(f"Meta IPCA + 4% a.a. em 2026: {meta:.2f}%")
res.assign(bateu_meta=res["no_ano_2026"] > meta)[["indice", "no_ano_2026", "bateu_meta"]]

Meta IPCA + 4% a.a. em 2026: 7.75%


,indice,no_ano_2026,bateu_meta
0,CDI,11.68,True
1,S&P 500 (US$),10.74,True
2,S&P 500 (R$),9.11,True
3,IMA-B,8.98,True
4,IRF-M,7.63,False
5,IHFA,5.81,False
6,IPCA,4.28,False
7,IBOV,4.22,False
8,Dólar,-1.47,False
9,IFIX,-3.41,False


## 6. Correlação entre as classes (12 meses)
Diversificar funciona quando os ativos não sobem e caem juntos.

In [5]:
largo.tail(12).corr().round(2)

indice,IBOV,S&P 500 (US$),S&P 500 (R$),IMA-B,CDI,IFIX,IRF-M,IPCA,IHFA,Dólar,BTC
indice,,,,,,,,,,,
IBOV,1.00,-0.10,-0.68,0.59,0.15,0.72,0.77,-0.02,0.49,-0.71,-0.27
S&P 500 (US$),-0.10,1.00,0.68,0.27,-0.28,0.20,0.35,-0.05,0.59,-0.43,0.27
S&P 500 (R$),-0.68,0.68,1.00,-0.18,-0.27,-0.26,-0.19,-0.22,0.21,0.37,0.41
IMA-B,0.59,0.27,-0.18,1.00,-0.31,0.44,0.77,0.19,0.61,-0.57,0.25
CDI,0.15,-0.28,-0.27,-0.31,1.00,0.15,-0.19,-0.10,-0.44,0.03,0.18
IFIX,0.72,0.20,-0.26,0.44,0.15,1.00,0.55,0.12,0.48,-0.57,-0.33
IRF-M,0.77,0.35,-0.19,0.77,-0.19,0.55,1.00,-0.05,0.83,-0.69,-0.08
IPCA,-0.02,-0.05,-0.22,0.19,-0.10,0.12,-0.05,1.00,-0.20,-0.19,-0.22
IHFA,0.49,0.59,0.21,0.61,-0.44,0.48,0.83,-0.20,1.00,-0.50,-0.04


## 7. Histórico de 15 anos
Retornos anuais de 2011 a 2025 (tabela "15 anos" da Meu Patrimônio) e 2026 composto dos meses já fechados. O **Bitcoin entra só a partir de 2020**, pela Mais Retorno (MarketVector Bitcoin Benchmark Rate, convertido para reais): antes disso o efeito dele num estudo de carteira é irrelevante, e juntar fontes diferentes distorce o acumulado. O "ao ano" usa só os meses que cada índice tem.

In [6]:
anuais, meses = historico.com_ano_parcial(historico.ler_anual(), largo)
per = historico.periodo(anuais, meses, 2011, 2026)
per

,acumulado_pct,ao_ano_pct,desde,meses
S&P 500 (R$),1862.28,20.68,2011.0,190.0
BTC,1351.74,47.92,2020.0,82.0
S&P 500 (US$),498.31,11.96,2011.0,190.0
IMA-B,394.84,10.63,2011.0,190.0
IRF-M,381.71,10.44,2011.0,190.0
IHFA,362.97,10.16,2011.0,190.0
CDI,347.25,9.92,2011.0,190.0
IFIX,261.17,8.45,2011.0,190.0
Dólar,227.97,7.79,2011.0,190.0
IPCA,141.33,5.72,2011.0,190.0


![acumulado no período](../outputs/acumulado_periodo.png)

In [7]:
historico.tabela_periodica(anuais, 2021, 2026)

,2021,2022,2023,2024,2025,2026
1,"BTC +72,3%","IHFA +13,7%","BTC +138,5%","BTC +181,7%","IBOV +34,0%","CDI +11,7%"
2,"S&P 500 (R$) +36,3%","CDI +12,4%","IBOV +22,3%","S&P 500 (R$) +57,7%","IFIX +21,2%","S&P 500 (R$) +9,1%"
3,"IPCA +10,1%","IRF-M +8,8%","IRF-M +16,5%","Dólar +27,9%","IRF-M +18,2%","IMA-B +9,0%"
4,"Dólar +7,4%","IMA-B +6,4%","IMA-B +16,1%","CDI +10,9%","IHFA +15,3%","IRF-M +7,6%"
5,"CDI +4,4%","IPCA +5,8%","IFIX +15,5%","IHFA +5,8%","CDI +14,3%","IHFA +5,8%"
6,"IHFA +2,0%","IBOV +4,7%","S&P 500 (R$) +15,3%","IPCA +4,8%","IMA-B +13,2%","IPCA +4,3%"
7,"IMA-B -1,3%","IFIX +2,3%","CDI +13,0%","IRF-M +1,9%","IPCA +4,3%","IBOV +4,2%"
8,"IRF-M -2,0%","Dólar -6,5%","IHFA +9,3%","IMA-B -2,4%","S&P 500 (R$) +3,5%","Dólar -1,5%"
9,"IFIX -2,3%","S&P 500 (R$) -24,6%","IPCA +4,6%","IFIX -5,9%","Dólar -11,1%","IFIX -3,4%"
10,"IBOV -11,9%","BTC -66,3%","Dólar -7,2%","IBOV -10,4%","BTC -17,2%","BTC -12,6%"


## 8. Carteira teórica
Pesos de exemplo, **rebalanceados no início de cada ano**: o retorno do ano é a média ponderada dos índices. Na renda fixa dá para informar a taxa obtida pelo cliente (x% do CDI, IPCA + x%, prefixado x% a.a.), que substitui o índice só na carteira; CDI, IPCA e a meta continuam sendo as referências de mercado.

In [8]:
cart = carteira.Carteira(carteira.EXEMPLO)
df = cart.anual(anuais, meses, 2011, 2026)
print(cart.resumo(df))
df

{'inicio': 2011, 'fim': 2026, 'meses': 190, 'acumulado_pct': 431.06482025772186, 'ao_ano_pct': 11.121678176859406, 'cdi_pct': 347.2544914201383, 'ipca_pct': 141.33274851578346, 'real_aa_pct': 5.107463142051771, 'meta_pct': 349.06664445786504}


,carteira,cdi,ipca,irfm,meses
ano,,,,,
2011,10.19,11.60,6.50,14.50,12
2012,17.04,8.40,5.80,14.30,12
2013,3.54,8.10,5.90,2.60,12
2014,10.25,10.80,6.40,11.40,12
2015,11.79,13.20,10.70,7.10,12
2016,19.26,14.00,6.30,23.40,12
2017,14.75,9.90,2.90,14.80,12
2018,8.91,6.40,3.70,10.30,12
2019,18.62,6.00,4.30,12.00,12


![carteira](../outputs/carteira_exemplo.png)

## 9. Risco: Sharpe comparado com o mercado
Volatilidade e Sharpe em **anos completos** (o ano parcial fica de fora). Um Sharpe sozinho diz pouco; por isso ele é comparado com o Sharpe de cada índice nos **mesmos anos**, e com a média dos índices. Ficam fora da comparação o CDI (é a referência), a inflação e o S&P em dólar.

In [9]:
print({k: round(v, 2) for k, v in cart.risco(df).items()})
tab = carteira.comparar_sharpe(cart, df, anuais)
print(f"média dos índices {tab.attrs['media_indices']:.2f} | carteira acima de {tab.attrs['acima_de']} de {tab.attrs['n_indices']}")
tab

{'anos': 15, 'vol_pct': 5.29, 'vol_cdi_pct': 3.59, 'excesso_medio_pp': 1.58, 'sharpe': 0.31, 'anos_abaixo_cdi': 7}
média dos índices 0.14 | carteira acima de 6 de 7


,sharpe
S&P 500 (R$),0.55
Carteira,0.31
IHFA,0.21
IRF-M,0.17
IMA-B,0.13
IFIX,0.03
Dólar,-0.01
IBOV,-0.12


![sharpe](../outputs/sharpe_mercado.png)

### Prefixado: taxa contratada, risco de mercado
Com taxa prefixada do cliente, a RF Pré rende a taxa todo ano, e a carteira parece não oscilar. Mas quem precisa vender antes do vencimento recebe o **preço de mercado**. Por isso, no risco, a oscilação do prefixado é a do IRF-M no período (marcação a mercado): soma-se o desvio do IRF-M em torno da sua média, e o retorno médio não muda.

In [10]:
pre = carteira.Carteira({"CDI": 70, "IRF-M": 30}, pre_aa=12)
dp = pre.anual(anuais, meses, 2011, 2026)
dp = dp[dp["meses"] == 12]   # risco em anos completos
pd.DataFrame({"sem marcação": dp["carteira"], "com marcação a mercado": pre.com_marcacao(dp)}).agg(["mean", "std"])

,sem marcação,com marcação a mercado
mean,10.42,10.42
std,2.52,4.01


## 10. Aportes anuais
Aporte inicial no começo do período e aporte anual no início de cada ano, inclusive o primeiro. A rentabilidade real compara o saldo final com os aportes corrigidos pelo IPCA desde a data de cada um.

In [11]:
s = carteira.Carteira.simular_aportes(df, inicial=50_000, anual=12_000)
{k: round(v, 2) for k, v in s.items()}

{'aportado': 242000.0,
 'saldo': 777882.03,
 'aportes_corrigidos_ipca': 429366.72,
 'ganho_bruto': 535882.03,
 'ganho_real': 348515.3,
 'bruto_pct': 221.44,
 'real_pct': 81.17}

## 11. Leitura e limitações
- Com 12 meses, volatilidade, Sharpe e correlação são **estimativas instáveis**: um único mês ruim muda bastante os números. Servem para comparar classes, não para prever.
- No histórico, o Sharpe usa um retorno por ano: com 5 anos completos são só 5 observações. É uma régua de comparação, não uma medida estatisticamente firme.
- Com Bitcoin na carteira, o estudo começa em 2020; nenhum período anterior é comparável.
- O IPCA do último mês pode ser a prévia (IPCA-15) até o IBGE divulgar o índice cheio; o pipeline marca esses valores.
- Sem custos, impostos ou taxas. Análise descritiva, não recomendação de investimento.